# COVID-19 TRA × TRB Biomarker Co-occurrence Analysis

For each paired donor, test whether the presence of a significant TRA clonotype
and a significant TRB clonotype co-occur more (or less) than expected by chance.

Significant clonotypes are those from `covid19_biomarkers.ipynb` (Fisher scan,
q < 0.05 after BH correction): 39 TRB + 4 TRA = 156 TRA × TRB pairs.

**Data sources**
- `tmp/fisher_trb.parquet` / `tmp/fisher_tra.parquet` — de-novo Fisher scan results  
- AIRR TSV files — one file per donor per locus in `airr_covid19/`  
- `notebooks/assets/large/airr_covid19/metadata.tsv` — donor metadata

## 1. Setup

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=None):
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / 'pyproject.toml').exists() and (cand / 'mir').exists():
            return cand
    raise FileNotFoundError('Cannot find repo root')

repo_root  = find_repo_root()
data_root  = repo_root / 'notebooks' / 'assets' / 'large' / 'airr_covid19'
tmp_dir    = repo_root / 'tmp'
assets_dir = repo_root / 'notebooks' / 'assets'

import importlib.metadata
print(f'Python {sys.version.split()[0]}  mirpy-lib {importlib.metadata.version("mirpy-lib")}')
print(f'numpy {np.__version__}  pandas {pd.__version__}')

## 2. Load Significant Biomarkers

In [ ]:
# Load de-novo Fisher scan results; keep only significant hits.
FDR_THRESHOLD = 0.05

fisher_trb = pd.read_parquet(tmp_dir / 'fisher_trb.parquet')
fisher_tra = pd.read_parquet(tmp_dir / 'fisher_tra.parquet')

sig_trb = fisher_trb[fisher_trb['p_value_adj'] < FDR_THRESHOLD][['junction_aa','log2_fe','p_value_adj']].copy()
sig_tra = fisher_tra[fisher_tra['p_value_adj'] < FDR_THRESHOLD][['junction_aa','log2_fe','p_value_adj']].copy()

sig_trb_list = sig_trb['junction_aa'].tolist()
sig_tra_list = sig_tra['junction_aa'].tolist()
sig_trb_set  = set(sig_trb_list)
sig_tra_set  = set(sig_tra_list)

N_PAIRS = len(sig_trb_list) * len(sig_tra_list)
print(f'Significant TRB clonotypes: {len(sig_trb_list)}')
print(f'Significant TRA clonotypes: {len(sig_tra_list)}')
print(f'TRA × TRB pairs to test:    {N_PAIRS}')
print('\nTRB biomarkers:')
print(sig_trb[['junction_aa','log2_fe','p_value_adj']].to_string(index=False))
print('\nTRA biomarkers:')
print(sig_tra[['junction_aa','log2_fe','p_value_adj']].to_string(index=False))

## 3. Per-Donor CDR3 Presence Scan

In [ ]:
# Load metadata and find paired donors (TRA + TRB).
meta = pd.read_csv(data_root / 'metadata.tsv', sep='\t', dtype={'donor_id': 'string'})
meta_filt = meta[meta['COVID_status'].isin(['COVID', 'healthy'])].copy()
meta_filt['locus_upper'] = meta_filt['locus'].str.upper()
trb_donors = set(meta_filt[meta_filt['locus_upper']=='TRB']['donor_id'])
tra_donors = set(meta_filt[meta_filt['locus_upper']=='TRA']['donor_id'])
paired_donors = sorted(trb_donors & tra_donors)

# COVID status per donor
meta_donor  = meta_filt[meta_filt['donor_id'].isin(paired_donors)]
covid_status = meta_donor.drop_duplicates('donor_id').set_index('donor_id')['COVID_status']
y_status     = {d: (1 if covid_status.get(d) == 'COVID' else 0) for d in paired_donors}

meta_by_locus = {
    'TRB': meta_filt[meta_filt['locus_upper']=='TRB'].set_index('donor_id'),
    'TRA': meta_filt[meta_filt['locus_upper']=='TRA'].set_index('donor_id'),
}

n_covid   = sum(1 for v in y_status.values() if v == 1)
n_healthy = sum(1 for v in y_status.values() if v == 0)
print(f'Paired donors: {len(paired_donors)}  (COVID={n_covid}, healthy={n_healthy})')

In [ ]:
# Scan AIRR files for presence of significant CDR3s.
# Result: presence_trb[donor] = frozenset of present sig TRB CDR3s
#         presence_tra[donor] = frozenset of present sig TRA CDR3s
presence_trb: dict[str, frozenset[str]] = {}
presence_tra: dict[str, frozenset[str]] = {}

t0 = time.perf_counter()
for locus_key, target_set, presence_dict in [
        ('TRB', sig_trb_set, presence_trb),
        ('TRA', sig_tra_set, presence_tra)]:
    if not target_set:
        for d in paired_donors:
            presence_dict[d] = frozenset()
        continue
    locus_meta = meta_by_locus[locus_key]
    for donor_id in paired_donors:
        if donor_id not in locus_meta.index:
            presence_dict[donor_id] = frozenset()
            continue
        path = data_root / str(locus_meta.loc[donor_id, 'file_name'])
        try:
            df   = pd.read_csv(path, sep='\t', usecols=['cdr3aa'], dtype=str)
            hits = frozenset(df['cdr3aa'].dropna()) & target_set
        except Exception:
            hits = frozenset()
        presence_dict[donor_id] = hits

elapsed = time.perf_counter() - t0
print(f'CDR3 presence scan: {elapsed:.1f}s')

# Quick sanity check
for cdr3 in sig_trb_list:
    n = sum(1 for d in paired_donors if cdr3 in presence_trb[d])
    print(f'  TRB  {cdr3:35s}  hits={n}')
for cdr3 in sig_tra_list:
    n = sum(1 for d in paired_donors if cdr3 in presence_tra[d])
    print(f'  TRA  {cdr3:35s}  hits={n}')

## 4. TRA × TRB Co-occurrence Fisher Tests

In [ ]:
# For each (TRA_cdr3, TRB_cdr3) pair: 2×2 Fisher test of co-occurrence.
# Run separately for all donors, COVID-only, and healthy-only.
EPS = 0.5
results = []
for tra_cdr3 in sig_tra_list:
    for trb_cdr3 in sig_trb_list:
        for subset_label, donor_subset in [
                ('all',     paired_donors),
                ('covid',   [d for d in paired_donors if y_status[d] == 1]),
                ('healthy', [d for d in paired_donors if y_status[d] == 0])]:

            has_tra = [cdr3 in presence_tra.get(d, frozenset()) for d in donor_subset for cdr3 in [tra_cdr3]]
            has_trb = [cdr3 in presence_trb.get(d, frozenset()) for d in donor_subset for cdr3 in [trb_cdr3]]
            # flatten (list comprehension above already flat)
            has_tra_arr = np.array([tra_cdr3 in presence_tra.get(d, frozenset()) for d in donor_subset])
            has_trb_arr = np.array([trb_cdr3 in presence_trb.get(d, frozenset()) for d in donor_subset])

            both = int((has_tra_arr & has_trb_arr).sum())
            tra_only = int((has_tra_arr & ~has_trb_arr).sum())
            trb_only = int((~has_tra_arr & has_trb_arr).sum())
            neither  = int((~has_tra_arr & ~has_trb_arr).sum())

            if (both + tra_only) == 0 or (both + trb_only) == 0:
                continue

            _, p = fisher_exact([[both, tra_only], [trb_only, neither]],
                                alternative='two-sided')

            freq_tra = (both + EPS) / (both + tra_only + 2*EPS)
            freq_notra = (trb_only + EPS) / (trb_only + neither + 2*EPS)
            fe = freq_tra / freq_notra

            results.append({
                'tra_cdr3':  tra_cdr3,
                'trb_cdr3':  trb_cdr3,
                'subset':    subset_label,
                'n':         len(donor_subset),
                'n_both':    both,
                'n_tra_only':tra_only,
                'n_trb_only':trb_only,
                'n_neither': neither,
                'freq_trb_given_tra': freq_tra,
                'freq_trb_given_no_tra': freq_notra,
                'fold_enrichment': fe,
                'log2_fe': float(np.log2(max(fe, 1e-6))),
                'p_value':  p,
            })

pair_df = pd.DataFrame(results)
# FDR correction within each subset independently
for subset in pair_df['subset'].unique():
    mask = pair_df['subset'] == subset
    _, padj, _, _ = multipletests(pair_df.loc[mask, 'p_value'], method='fdr_bh')
    pair_df.loc[mask, 'p_value_adj'] = padj

pair_df['neg_log10_padj'] = -np.log10(np.clip(pair_df['p_value_adj'], 1e-300, None))
pair_df = pair_df.sort_values(['subset','p_value_adj']).reset_index(drop=True)

for sub in ['all','covid','healthy']:
    n_sig = int((pair_df[(pair_df['subset']==sub)]['p_value_adj'] < FDR_THRESHOLD).sum())
    print(f'{sub:8s}  pairs tested: {(pair_df["subset"]==sub).sum():4d}  significant: {n_sig}')
print()
print(pair_df[pair_df['subset']=='all'].head(20)[['tra_cdr3','trb_cdr3','n_both','log2_fe','p_value_adj']].to_string(index=False))

## 5. Co-occurrence Heatmaps

In [ ]:
sns.set_theme(style='white', context='talk')

def _make_heatmap(df_sub, title_suffix, filepath):
    pivot_fe = df_sub.pivot_table(
        index='tra_cdr3', columns='trb_cdr3', values='log2_fe', aggfunc='first')
    pivot_p  = df_sub.pivot_table(
        index='tra_cdr3', columns='trb_cdr3', values='p_value_adj', aggfunc='first')

    annot = pivot_p.applymap(lambda v: ('*' if v < FDR_THRESHOLD else ''))

    figw = max(8, pivot_fe.shape[1] * 0.9 + 3)
    figh = max(4, pivot_fe.shape[0] * 0.8 + 2)
    fig, ax = plt.subplots(figsize=(figw, figh))

    sns.heatmap(
        pivot_fe, cmap='RdBu_r', center=0, ax=ax,
        annot=annot, fmt='', annot_kws={'fontsize': 11, 'color': 'black'},
        linewidths=0.5, cbar_kws={'label': 'log₂FE (TRB | TRA)'}
    )
    ax.set_title(f'TRA × TRB Co-occurrence — {title_suffix}\n'
                 f'(*: q<{FDR_THRESHOLD}, log₂FE>0 = co-enriched)',
                 fontweight='bold')
    ax.set_xlabel('TRB CDR3');  ax.set_ylabel('TRA CDR3')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(filepath, bbox_inches='tight')
    plt.show()

# All donors
_make_heatmap(
    pair_df[pair_df['subset']=='all'],
    f'All donors (n={len(paired_donors)})',
    assets_dir / 'covid19_pairing_all.pdf'
)

# COVID donors only
n_covid_donors = sum(1 for v in y_status.values() if v == 1)
_make_heatmap(
    pair_df[pair_df['subset']=='covid'],
    f'COVID donors (n={n_covid_donors})',
    assets_dir / 'covid19_pairing_covid.pdf'
)

# Healthy donors only
n_healthy_donors = sum(1 for v in y_status.values() if v == 0)
_make_heatmap(
    pair_df[pair_df['subset']=='healthy'],
    f'Healthy donors (n={n_healthy_donors})',
    assets_dir / 'covid19_pairing_healthy.pdf'
)

## 6. Bubble Chart — Co-occurrence Landscape

In [ ]:
# Bubble chart: x=TRA biomarker, y=TRB biomarker; size=n_both; color=log2_fe;
# mark significant pairs with a black border.
df_all = pair_df[pair_df['subset']=='all'].copy()
fig, ax = plt.subplots(figsize=(max(7, len(sig_tra_list)*2), max(8, len(sig_trb_list)*0.7)))

tra_pos = {c: i for i, c in enumerate(sig_tra_list)}
trb_pos = {c: i for i, c in enumerate(sig_trb_list)}

import matplotlib.colors as mcolors
cmap   = plt.get_cmap('RdBu_r')
norm   = mcolors.TwoSlopeNorm(vcenter=0,
                               vmin=df_all['log2_fe'].min(),
                               vmax=df_all['log2_fe'].max())
max_n  = df_all['n_both'].max() or 1

for _, row in df_all.iterrows():
    x   = tra_pos[row['tra_cdr3']]
    y   = trb_pos[row['trb_cdr3']]
    sz  = 50 + 500 * (row['n_both'] / max_n)
    col = cmap(norm(row['log2_fe']))
    ec  = 'black' if row['p_value_adj'] < FDR_THRESHOLD else 'none'
    lw  = 1.5     if row['p_value_adj'] < FDR_THRESHOLD else 0
    ax.scatter(x, y, s=sz, c=[col], edgecolors=ec, linewidths=lw, alpha=0.85, zorder=3)

ax.set_xticks(range(len(sig_tra_list)))
ax.set_xticklabels(sig_tra_list, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(sig_trb_list)))
ax.set_yticklabels(sig_trb_list, fontsize=8)
ax.set_xlabel('TRA biomarker CDR3', fontsize=11)
ax.set_ylabel('TRB biomarker CDR3', fontsize=11)
ax.set_title('TRA × TRB Co-occurrence Bubble Chart\n'
             '(size=n_both; color=log₂FE; black border=q<0.05)',
             fontweight='bold')
ax.grid(True, ls=':', alpha=0.5, zorder=0)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='log₂FE', shrink=0.5)
plt.tight_layout()
plt.savefig(assets_dir / 'covid19_pairing_bubble.pdf', bbox_inches='tight')
plt.show()

## 7. Summary

In [ ]:
print('=== TRA × TRB Co-occurrence Summary ===')
print(f'  Significant TRB biomarkers: {len(sig_trb_list)}')
print(f'  Significant TRA biomarkers: {len(sig_tra_list)}')
print(f'  TRA × TRB pairs tested: {N_PAIRS}')
print()

for sub in ['all', 'covid', 'healthy']:
    sub_df = pair_df[pair_df['subset'] == sub]
    sig_mask = sub_df['p_value_adj'] < FDR_THRESHOLD
    print(f'  Subset={sub}: {sig_mask.sum()} significant pairs')
    if sig_mask.sum() > 0:
        print(sub_df[sig_mask][['tra_cdr3','trb_cdr3','n_both','log2_fe','p_value_adj']].to_string(index=False))
    print()

# Save full results
pair_df.to_csv(repo_root / 'tmp' / 'pairing_cooccurrence.csv', index=False)
print('Full co-occurrence results saved to tmp/pairing_cooccurrence.csv')